In [101]:
%reset -f

# Transform HTML Code to TXT

In [102]:
import os, re, html, pandas as pd
from pathlib import Path
from html import unescape


excel_path = "/Users/panglinshao/Desktop/IPO/S-1/S-1 filings/S-1_sections_extraction/garbled_s1.xlsx"  
root_dir   = "/Users/panglinshao/Desktop/IPO/S-1/S-1 filings/S-1_merge"           
out_dir    = "/Users/panglinshao/Desktop/IPO/S-1/S-1 filings/htmlcode_txt"
txt_dir = Path(out_dir) / "txt"
txt_dir.mkdir(parents=True, exist_ok=True)       

Path(out_dir).mkdir(parents=True, exist_ok=True)


### read CIK

In [103]:
df = pd.read_excel(excel_path, dtype=str)
df.columns = df.columns.str.strip().str.lower()
assert 'cik' in df.columns, "Excel needs a column named CIK"
cik_list = df['cik'].dropna().str.strip().tolist()

def read_text(path):
    return Path(path).read_text(encoding="utf-8", errors="ignore")

def filed_as_of_date(raw:str):
    m = re.search(r"FILED\s+AS\s+OF\s+DATE:\s*(\d{8})", raw, flags=re.IGNORECASE)
    return m.group(1) if m else None

### text cleaning

In [ ]:
# replace the linebreak within <>...</> with a space
def collapse_intra_tag_newlines(html: str) -> str:
    return re.sub(
        r">(.*?)<",
        lambda m: ">" + re.sub(r"[ \t]*\n+[ \t]*", " ", m.group(1)) + "<",
        html,
        flags=re.S
    )

def strip_tags_preserve_br(s):
    s = re.sub(r"<\s*BR\s*/?>", "\n", s, flags=re.IGNORECASE) # replace wrap tag
    s = re.sub(r"<\s*LI\s*>", "• ", s, flags=re.IGNORECASE)  # replace list item start tag
    s = re.sub(r"</\s*TABLE\s*>", "\n\n", s, flags=re.IGNORECASE)        
    s = re.sub(r"</\s*(DIV|P|CENTER|H[1-6]|UL|OL|TR|TD|TH|LI)\s*>", "\n", s, flags=re.IGNORECASE)  # replace block end tag with wrap tag
    s = re.sub(r"<(?!/?PAGE\b)[^>]+>", "", s)  # drop all tags except PAGE tag
    s = html.unescape(s)  # transform html entities to original characters
    s = re.sub(r"[ \t]+\n", "\n", s)  # remove excess spaces/tabs at the end of lines
    s = re.sub(r"\n[ \t]+", "\n", s)  # remove excess spaces/tabs at the beginning of lines
    s = re.sub(r"[ \t]+", " ", s)  # fold consecutive spaces/tabs within a line into a single space
    return s.strip()

def is_page_token(tok: str) -> bool:
    tok = tok.strip()
    if not tok:
        return False
    return bool(re.match(r"^\d+$", tok))  # only identify pure digits as page number

def convert_toc_table(rows):
    items = []
    for row in rows:
        cells = [c.strip() for c in row if c.strip()]  # extract row containing information after strip
        if not cells: continue
        page, title = "", ""
        for c in reversed(cells):  # locate page from right to left
            if is_page_token(c):
                page = c; break
        if page:
            idx = next((i for i,c in enumerate(cells) if c==page), len(cells))  # locate page index from left to right
            title = " ".join(cells[:idx]).strip()  # extract title
        else:
            title = " ".join(cells).strip()
        if title and title.lower()!="page":  # TBR
            items.append((title, page))
    cap = min(max([len(t) for t,p in items] + [0]), 90)  
    out = ["TABLE OF CONTENTS",""]
    for t,p in items:
        t = t[:cap]
        col = 95
        pad = max(1, col - len(t) - len(p))
        out.append(f"{t}{' ' * pad}{p}")
    return "\n".join(out) + "\n<TOC end>"

def parse_table(table_html):
    table_html = re.sub(r"</?THEAD[^>]*>", "", table_html, flags=re.IGNORECASE)
    table_html = re.sub(r"</?TBODY[^>]*>", "", table_html, flags=re.IGNORECASE)
    table_html = re.sub(r"</?TFOOT[^>]*>", "", table_html, flags=re.IGNORECASE)
    row_htmls = re.findall(r"<TR\b[^>]*>(.*?)</TR>", table_html, flags=re.IGNORECASE|re.DOTALL)  # capture rows in tables
    rows = []
    for rh in row_htmls:
        cells = re.findall(r"<T[DH]\b[^>]*>(.*?)</T[DH]>", rh, flags=re.IGNORECASE|re.DOTALL) # capture <th> and <td> (non-greedy)
        clean_cells = []
        for c in cells:
            c_clean = strip_tags_preserve_br(c)
            c_clean = c_clean.replace("\u00A0", " ")
            c_clean = c_clean.strip()
            if c_clean:             
                clean_cells.append(c_clean)
        if clean_cells:                
            rows.append(clean_cells)
    return rows

def _is_standalone_toc_before(text: str, table_start: int, window: int = 300) -> bool:
    ctx_raw = text[max(0, table_start - window):table_start]  # capture 300 characters before tables
    ctx = collapse_intra_tag_newlines(ctx_raw)  # collapse linebreak within <>...</>
    ctx = re.sub(r"<table\b[^>]*>.*?</table>", "", ctx, flags=re.I | re.S)  # drop all <TABLE>...</TABLE>
    # list all tags except <H1-6>
    for tag, inner in re.findall(
        r"<(?!h[1-6]\b)([a-z][\w:-]*)\b[^>]*>(.*?)</\s*\1\s*>",
        ctx, flags=re.I | re.S
    ):
        # extract body text
        norm = re.sub(r"<[^>]+>", " ", inner)
        norm = html.unescape(norm)
        norm = re.sub(r"\s+", " ", norm).strip().lower()

        if norm == "table of contents":
            return True
    return False

def convert_table_rows_to_paragraphs(rows):
    def clean_cell(html):
        if not html:
            return ""
        s = str(html)
        s = re.sub(r"<\s*br\s*/?>", "\n", s, flags=re.I)  # replace <br> with \n
        s = re.sub(r"<[^>]+>", " ", s)  # drop other tags
        s = unescape(s)  # decode html entities
        s = re.sub(r"[ \t\r\f\v]+", " ", s)
        s = re.sub(r" *\n *", "\n", s)
        s = s.strip()
        return s

    paras = []
    for row in (rows or []):
        cells = row or []
        texts = [t for t in (clean_cell(c) for c in cells) if t]
        if not texts:
            continue
        line = " ".join(texts).strip()
        if line:
            paras.append(line)

    return "\n\n".join(paras)


NUMERIC_CELL_RE = re.compile(r"""
    ^\s*
    (?:[$€£])?            # currency symbol
    \(?[+-]?\d{1,3}(?:,\d{3})*|\(?[+-]?\d+   # thousands-grouped or plain integer
    (?:\.\d+)?            # decimal part
    \)?                   # closing parenthesis
    \s*(?:%|bps|bp|x)?    # suffix: percent / basis points / multiplier
    \s*$""", re.IGNORECASE | re.VERBOSE)

def is_numeric_like_cell(s: str) -> bool:
    if not s:
        return False
    if NUMERIC_CELL_RE.fullmatch(s):
        return True
    return False


def render_first_text_block(raw, *, report_flags=False):
    m = re.search(r"<TEXT>(.*?)</TEXT>", raw, flags=re.DOTALL|re.IGNORECASE)
    if not m: raise RuntimeError("No <TEXT> block found.")
    text_block = m.group(1)
    text_block = collapse_intra_tag_newlines(text_block)
    toc_found = False

    # drop <H5>...</H5>
    text_block = re.sub(
    r"<\s*H5\b[^>]*>\s*(?:<[^>]*>\s*)*"
    r"Table(?:\s|&nbsp;|<[^>]*>)+of(?:\s|&nbsp;|<[^>]*>)+Contents"
    r"(?:\s*<[^>]*>)*\s*</\s*H5\s*>",
    "",
    text_block,
    flags=re.I|re.S
    )
    
    # replace html page tag with <PAGE> without dropping body text within it
    text_block = re.sub(
        r"<[a-z][\w:-]*\b[^>]*\bstyle\s*=\s*['\"][^'\"]*"
        r"(?:page-break-before\s*:\s*always|break-before\s*:\s*page)"
        r"[^'\"]*['\"][^>]*>",
        "\n<PAGE>\n",
        text_block, flags=re.I
    )

    text_block = re.sub(r"<!--\s*pagebreak\s*-->", "\n<PAGE>\n", text_block, flags=re.I)

    # drop images and <HEAD>...</HEAD>
    text_block = re.sub(r"<IMG\b[^>]*>", "", text_block, flags=re.I)  
    text_block = re.sub(r"<HEAD\b.*?</HEAD>", "", text_block, flags=re.I|re.S)  

    # identify and render tables
    table_placeholder = "§§TABLE_PLACEHOLDER_{}§§"
    table_blocks = {}
    tables = list(re.finditer(r"<TABLE\b[^>]*>.*?</TABLE>", text_block, 
                              flags=re.IGNORECASE|re.DOTALL))

    ALLOWED_BETWEEN = re.compile(
        r"(?is)^(?:\s|&nbsp;|</?(?:div|span|center|p|font)[^>]*>)*$"
    )

    consumed = set()  # original-order indices of tables merged into a previous one
    n = len(tables)
    
    for idx, tm in enumerate(reversed(tables)):
        oidx = n - 1 - idx
        if oidx in consumed:
            continue

        thtml = tm.group(0)  # extract the whole html code of tables
        start, end = tm.start(), tm.end()
        rows = parse_table(thtml)

        toc_like = _is_standalone_toc_before(text_block, start, window=300)

        # merge adjacent tables that follow and are only separated by blank space or layout tags
        if toc_like:
            
            j = oidx + 1
            while j < n:
                if j in consumed:
                    j += 1
                    continue
                next_m = tables[j]
                between = text_block[end:next_m.start()]

                # allowed interlayers: blank/&nbsp;/pure layout tags
                if not ALLOWED_BETWEEN.fullmatch(between or ""):
                    break

                rows2 = parse_table(next_m.group(0))
                rows.extend(rows2)
                end = next_m.end()
                consumed.add(j)
                j += 1


        if toc_like:
            toc_found = True
            ttext = convert_toc_table(rows)
            key = table_placeholder.format(idx)
            table_blocks[key] = ttext
            text_block = text_block[:start] + f"\n{key}\n\n" + text_block[end:]
        else:
            # drop tables with over 30% of body text being number
            cells_flat = [c for row in (rows or []) for c in (row or []) if c]
            total_cells = len(cells_flat)
            if total_cells:
                numeric_cells = sum(1 for c in cells_flat if is_numeric_like_cell(c))
                if numeric_cells / total_cells >= 0.3:
                    text_block = text_block[:start] + text_block[end:]
                    continue

            gtext = convert_table_rows_to_paragraphs(rows)
            key = table_placeholder.format(f"GEN_{idx}")
            table_blocks[key] = gtext
            text_block = text_block[:start] + f"\n{key}\n\n" + text_block[end:]
    
    # drop HTML tags
    text_block = re.sub(r"<HR\b[^>]*>", "\n", text_block, flags=re.IGNORECASE)
    text_block = strip_tags_preserve_br(text_block)
    
    # drop <PAGE>
    text_block = re.sub(r"\n*<\s*/?\s*PAGE\s*>\n*", "", text_block, flags=re.IGNORECASE)

    # drop "Table of Contents" in one exclusive line
    text_block = re.sub(
    r"(?im)^\s*table\s+of\s+contents\s*[:\-–—]?\s*$",
    "", text_block
    )

    # drop excess blank lines
    text_block = re.sub(r"\n{4,}", "\n\n\n", text_block).strip()

    # replace placeholder with toc text
    for key, ttext in table_blocks.items():
        text_block = text_block.replace(key, ttext.rstrip()+"\n\n")

    # formatting
    #text_block = re.sub(r"\n{4,}", "\n\n\n", text_block).strip()
   
    final_out = "<TEXT>\n" + text_block + "\n</TEXT>\n"
    if report_flags:
        return final_out, {"found_toc": toc_found}
    return final_out

### locate S-1 file

In [105]:
def find_cik_folder(cik:str):
    target = cik.lstrip("0")
    candidates = []
    for name in os.listdir(root_dir):
        p = Path(root_dir)/name
        if p.is_dir() and name.lstrip("0")==target:
            candidates.append(p)
    return candidates[0] if candidates else None

def pick_earliest_s1(cik_dir:Path):
    best = None
    for path in cik_dir.rglob("*.txt"):
        raw = read_text(path)
        date = filed_as_of_date(raw)
        if not date:
            continue
        tup = (date, path, raw)
        if best is None or date < best[0]:
            best = tup
    return best  # (date, path, raw) or None

def pad_cik10(x: str) -> str:
    return str(x).strip().zfill(10)

### main

In [106]:
garbled = []

HEAD = 10
subset = cik_list[:HEAD]

for i, cik in enumerate(cik_list, 1):
    cik10 = pad_cik10(cik)

    folder = find_cik_folder(cik)
    if not folder:
        reason = "no CIK filefolder"
        garbled.append((cik10, reason))
        print(f"[{i:02d}/10 SKIP] {cik} -> {reason}")
        continue

    picked = pick_earliest_s1(folder)
    if not picked:
        reason = "not S-1/S-1A or <FILED AS OF DATE>"
        garbled.append((cik10, reason))
        print(f"[{i:02d}/10 SKIP] {cik} -> {reason}")
        continue
    date_str, src_path, raw = picked
    try:
        rendered, flags = render_first_text_block(raw, report_flags=True)
        
        if not flags.get("found_toc", False):
            reason = "not find TOC"
            garbled.append((cik10, reason))
            print(f"[{i:02d}/10 SKIP] {cik} -> {src_path.name} ({date_str}) -> {reason}")
            continue  
    
        out_path = txt_dir / f"{pad_cik10(cik)}.txt"
        out_path.write_text(rendered, encoding="utf-8")
        print(f"[{i:02d}/10 OK] {cik} -> {src_path.name} ({date_str}) -> {out_path}")
    
    except Exception as e:
        reason = f"error: {e}"
        garbled.append((cik10, reason))
        print(f"[{i:02d}/10 ERROR] fails to process {cik}: {e}")

garbled_path = Path(out_dir) / "garbled.txt"
with open(garbled_path, "w", encoding="utf-8") as f:
    for cik10, reason in garbled:
        f.write(f"{cik10}\t{reason}\n")

print(f"[REPORT] garbled written: {garbled_path} ({len(garbled)} items)")

[01/10 OK] 0000018169 -> 0000950123-09-034957.txt (20090814) -> /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/htmlcode_txt/txt/0000018169.txt
[02/10 OK] 0000029806 -> 0000950109-02-003159.txt (20020523) -> /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/htmlcode_txt/txt/0000029806.txt
[03/10 OK] 0000054003 -> 0001193125-04-164965.txt (20041001) -> /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/htmlcode_txt/txt/0000054003.txt
[04/10 SKIP] 0000101990 -> 0000950134-97-005395.txt (19970718) -> not find TOC
[05/10 OK] 0000707388 -> 0000921895-19-000716.txt (20190312) -> /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/htmlcode_txt/txt/0000707388.txt
[06/10 OK] 0000720154 -> 0001144204-11-011550.txt (20110228) -> /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/htmlcode_txt/txt/0000720154.txt
[07/10 OK] 0000721693 -> 0001213900-19-009569.txt (20190524) -> /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/htmlcode_txt/txt/0000721693.txt
[08/10 OK] 0000740761 -> 0001047469-04-011363.txt (20040408) ->

In [90]:
%reset -f

# Target sections extraction

In [91]:
import re, sys, datetime, traceback
from pathlib import Path
from typing import List, Dict, Optional, Callable, Tuple
import pandas as pd


BASE_DIR = Path.cwd()
S1_ROOT = Path("/Users/panglinshao/Desktop/IPO/S-1/S-1 filings/htmlcode_txt/txt")
OUT_DIR  = BASE_DIR / "S-1_sections_extraction"; OUT_DIR.mkdir(exist_ok=True, parents=True)
CHAPTER_DIR = OUT_DIR / "chapters"; CHAPTER_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
TARGET_SECTIONS = [
    "prospectus summary",
    "risk factors",
    "use of proceeds",
    "management's discussion and analysis",
    "business",
    "management",
]

SYNONYMS = {
    # Summary
    "prospectus summary": [
        "summary",
        "summary of prospectus",
    ],

    # Risk Factors
    "risk factors": [
        "risk factor",
    ],

    # Use of Proceeds
    "use of proceeds": [
        "use of the proceeds",
        "use of offering proceeds",
        "use of net proceeds",
        "application of proceeds",
    ],

    # MD&A
    "management's discussion and analysis": [
        "management's discussion and analysis of financial condition and results of operations",
        "management's discussion and analysis of financial condition and results of operation",
        "managements discussion and analysis of financial condition and results of operations",
        "md&a",
    ],

    # Business
    "business": [
        "our business",
        "business overview",
        "business of the company",
    ],

    # Management
    "management": [
        "directors and executive officers",
        "management and directors",
        "management, directors and executive officers",
        "directors, executive officers and key employees",
        "management and key employees",
    ],
}

### Basic tools

In [93]:
def read_text_safely(path: Path) -> str:
    try:
        txt = path.read_text(encoding="utf-8")
        print(f"[INFO] file {path} is successfully read using utf-8, length {len(txt)}")
        return txt
    except UnicodeDecodeError:
        txt = path.read_text(encoding="latin-1", errors="ignore")
        print(f"[WARN] file {path} is successfully read using latin-1, length {len(txt)}")
        return txt
    
def strip_html(s: str) -> str:
    s = re.sub(r"(?is)<(script|style)[^>]*>.*?</\1>", " ", s)  # match and remove <script>…</script> and <style>…</style> code blocks
    s = re.sub(r"(?is)<[^>]+>", " ", s)  # remove html tags
    s = re.sub(r"[ \t\u00A0\u3000]+", " ", s)  # collapse multiple spaces, tabs, and non-breaking spaces (\u00A0) into a single regular space
    s = re.sub(r"\n{2,}", "\n", s).strip()
    return s

# simple normalization
def _canon(s: str) -> str:  
    return re.sub(r"\s+", " ", s.lower().replace("’", "'").replace("‘", "'")).strip()

# strict normalization
def canon(s: str) -> str:  
    s = s.lower().replace("’", "'").replace("‘", "'")
    s = re.sub(r"[^a-z0-9'\s&/-]+", " ", s) 
    s = re.sub(r"[\s\u00A0\u2000-\u200B\u202F\u205F\u3000]+", " ", s).strip()
    return s


DEBUG = False

def dbg(*args, flush=True):
    if DEBUG:
        print("[DBG]", *args, flush=flush)

def _build_canon_target_map(TARGET_SECTIONS, SYNONYMS):
    m = {}
    for base in TARGET_SECTIONS:
        m[canon(base)] = base
        for alt in SYNONYMS.get(base, []):
            m[canon(alt)] = base
    return m

### Extract TOC

In [94]:
def extract_first_toc_block(text: str, max_fallback_chars: int = 20000):

    m_start = re.search(r'(?im)^\s*table\s+of\s+contents\s*$', text)  # search the standalone "table of contents" and store the first matching object in m
    if not m_start: return None
    start = m_start.start()  # record the starting index of the table-of-contents heading in the full text

    m_end = re.search(r'(?im)^[ \t]*<\s*TOC\s*end\s*>[ \t]*$', text[m_start.end():])
    if m_end:
        end = m_start.end() + m_end.start()
        return text[start:end]
    
    return text[start:start + max_fallback_chars]
  
    
def _pick_target_for_toc_title(toc_title: str, canon_map: Dict[str, str]):
    key = _canon(toc_title)
    return (canon_map[key], "exact") if key in canon_map else (None, "")


### Parse TOC and split pages

In [95]:
def parse_toc_entries(toc_block: str) -> List[Dict]:

    entries: List[Dict] = []
    pat = re.compile(r"^(.+?)\s+(\d{1,3})$")

    for raw in toc_block.splitlines():
        line = raw.strip()
        if not line:
            continue
        low = line.lower()
        if low == "table of contents":
            continue
        if low == "<toc end>":
            break

        m = pat.match(line)
        if not m:
            continue
        title = m.group(1).strip()
        page = int(m.group(2))
        entries.append({"toc_title": title, "start_page": page})

    for i, e in enumerate(entries):
        e["end_page"] = entries[i+1]["start_page"] if i + 1 < len(entries) else None
   
    return entries


def split_into_pages(txt: str) -> List[Tuple[Optional[int], str]]:
    txt_norm = re.sub(r"\r\n?", "\n", txt)
    lines = txt_norm.split("\n")

    # capture standalone pure number or -num- or - num -
    def parse_page_marker(s: str) -> Optional[int]:
        s = s.strip()
        return int(s) if re.fullmatch(r"\d{1,3}", s) else None

    # collect the line indecies and number in all page lines
    num_markers: List[Tuple[int, int]] = []
    for idx, ln in enumerate(lines):
        n = parse_page_marker(ln)
        if n is not None:
            num_markers.append((idx, n))

    if not num_markers:
        print("[ERR] No page-number lines detected")
        raise RuntimeError("NO_PAGE_NUMBER_MARKERS")

    pages: List[Tuple[Optional[int], str]] = []

    # content before the first page number line -> page 1
    first_idx, first_num = num_markers[0]
    head_seg = "\n".join(lines[:first_idx]).strip("\n")
    pages.append((first_num, head_seg))

    # the content between two adjacent page number rows belongs to the right page number
    for (i_idx, i_num), (j_idx, j_num) in zip(num_markers, num_markers[1:]):
        seg_lines = lines[i_idx + 1 : j_idx]  # not include page lines
        page_text = "\n".join(seg_lines).strip("\n")
        pages.append((j_num, page_text))
    
    # content after the last page number -> last page
    last_idx, last_num = num_markers[-1]
    tail_seg = "\n".join(lines[last_idx + 1 :]).strip("\n")
    if tail_seg:
        pages.append((last_num, tail_seg))

    return pages

### Locate headings

In [96]:
# find the standalone heading
def _find_heading(page_text: str, heading: str) -> Optional[Tuple[int, int]]:
    pat1 = re.compile(rf"(?i)(^|\r?\n)[ \t]*{re.escape(heading)}[ \t]*(\r?\n)")
    m = pat1.search(page_text)
    if m:
        return (m.start(), m.end())

# find the heading, allow linebreaks within the heading
def _find_heading_grow_prefix_and_join_next_line(
    page_text: str,
    heading: str,
    *,
    min_effective_len: int = 1,
    context_window: int = 200,
):
    print(f"[grow] ENTER heading={repr(heading)}", flush=True)

    if not heading:
        print("[grow] ABORT: empty heading", flush=True)
        return None

    target_c = _canon(heading)
    if not target_c:
        print("[grow] ABORT: empty canon target", flush=True)
        return None

    lines = []
    offsets = []  # record the indecies of (start, end)
    pos = 0
    n = len(page_text)
    while pos <= n:
        nl = page_text.find("\n", pos)
        if nl == -1:  # cannot find "\n", meaning from pos to the end is the last line
            start, end = pos, n
            lines.append(page_text[start:end])
            offsets.append((start, end))
            break
        else:
            start, end = pos, nl + 1  
            lines.append(page_text[start:end])
            offsets.append((start, end))
            pos = nl + 1
    
    # match prefix
    best_idx = -1
    best_len = -1
    for i, (raw, (st, ed)) in enumerate(zip(lines, offsets)):
        line_core = raw.rstrip("\r\n")
        line_c = _canon(line_core)
        if not line_c:
            continue
        if len(line_c) < min_effective_len:
            continue
        if target_c.startswith(line_c):
            print(f"[grow] i={i} len={len(line_c)} line_c={repr(line_c)[:200]}", flush=True)
            if len(line_c) > best_len:
                best_len = len(line_c)
                best_idx = i  # update line index
    if best_idx < 0:
        return None
    
    # joint the next line
    st, ed = offsets[best_idx]
    cur_line_core = lines[best_idx].rstrip("\r\n").strip()
    if best_idx + 1 >= len(lines):
        return None
    nst, ned = offsets[best_idx + 1]
    nxt_line_core = lines[best_idx + 1].rstrip("\r\n").strip()
    joined = f"{cur_line_core} {nxt_line_core}"
    joined_c = _canon(joined)

    if joined_c == target_c:
        ctx_end = min(len(page_text), ned + context_window)
        context = page_text[st:ctx_end].replace("\r", "")
        print("[grow] OK", repr(heading), "context=", repr(context), flush=True)
        return (st, ned)

    return None


def _find_heading_two_phase(page_text: str, heading: str, *,
    report=None):
    
    def log(msg: str):
        if report:
            report(msg)
        else:
            print(msg, flush=True)

    span = _find_heading(page_text, heading)
    if span is not None:
        st, ed = span
        snippet = page_text[max(0, st-40): min(len(page_text), ed+40)]
        log(f"[two-phase] phase-1 exact-line matched; skip grow. where={st}-{ed} context={snippet!r}")
        return span
    
    log(f"[two-phase] phase-1 missed; try grow. heading={heading!r}")
    return _find_heading_grow_prefix_and_join_next_line(
        page_text,
        heading,
        min_effective_len=1,
    )

### Extract sections using pages

In [97]:
def extract_section_strict_by_printed_pages(
    pages_with_nums: List[Tuple[Optional[int], str]],
    this_ent: Dict,     # {"toc_title": str, "start_page": int}
    next_ent: Optional[Dict]  # {"toc_title": str, "start_page": int}
) -> str:
    
    printed_to_idx: Dict[int, int] = {}
    for idx, (num, _) in enumerate(pages_with_nums):
        if num is not None and num not in printed_to_idx:
            printed_to_idx[num] = idx

    hdr = this_ent["toc_title"]
    start_printed = int(this_ent["start_page"])
    start_idx = printed_to_idx.get(start_printed, None)
    if start_idx is None:
        return ""  

    start_page_text = pages_with_nums[start_idx][1]

    span = _find_heading_two_phase(start_page_text, hdr)

    if span is None:
        return ""  # if start pages does not contain the standalone target title or title with linebreaks

    _, start_heading_end = span
    parts: List[str] = []
    parts.append(start_page_text[start_heading_end:])

    # extract content until the end of the file if there is no next_ent
    if not next_ent:
        for j in range(start_idx + 1, len(pages_with_nums)):
            parts.append(pages_with_nums[j][1])
        return "\n".join(parts).strip()

    # determine the end title and end page when there is next_ent
    next_hdr = next_ent["toc_title"]
    end_printed = int(next_ent["start_page"])
    end_idx = printed_to_idx.get(end_printed, None)

    # if there is no end page in toc, extract content until the end of the file
    if end_idx is None:
        for j in range(start_idx + 1, len(pages_with_nums)):
            parts.append(pages_with_nums[j][1])
        return "\n".join(parts).strip()

    # add body text after the start page and before the end page
    for j in range(start_idx + 1, end_idx):
        parts.append(pages_with_nums[j][1])

    end_page_text = pages_with_nums[end_idx][1]
    end_span = _find_heading_two_phase(end_page_text, next_hdr)
    

    if end_idx == start_idx:
        if end_span is not None:
            end_heading_start, _ = end_span
            return end_page_text[start_heading_end:end_heading_start].strip()
        else:
            return end_page_text[start_heading_end:].strip()
    
    if end_span is not None:
        end_heading_start, _ = end_span
        parts.append(end_page_text[:end_heading_start])
    else:
        parts.append(end_page_text)

    return "\n".join(parts).strip()


def locate_sections(
    full_txt: str,
    all_toc_entries: List[Dict],
    target_toc_entries: List[Dict],
) -> List[Dict]:

    pages_with_nums: List[Tuple[Optional[int], str]] = split_into_pages(full_txt)

    sections: List[Dict] = []

    def _canon_title(x: str) -> str:
        return re.sub(r"\s+", " ", x or "").strip().lower()

    # contruct indecies for toc
    index_map: Dict[Tuple[int, str], int] = {}
    for i, ent in enumerate(all_toc_entries):
        if "start_page" in ent and "toc_title" in ent:
            try:
                key = (int(ent["start_page"]), _canon_title(ent["toc_title"]))
                if key not in index_map:
                    index_map[key] = i
            except Exception:
                pass

    # extract target sections
    for t in target_toc_entries:
        raw_heading = t.get("toc_title")
        canon_heading = _canon_title(raw_heading)
        raw_start = t.get("start_page")
        print(f"[TARGET] section={t.get('section')}  heading={raw_heading!r}  canon={canon_heading!r}  start_page={raw_start!r}")
        
        try:
            t_key = (int(t["start_page"]), _canon_title(t["toc_title"]))
        except Exception:  # if t is missing or page cannot be int, return ""
            sections.append({"section": t.get("section"), "heading": t.get("toc_title"), "text": ""})
            continue

        idx_in_all = index_map.get(t_key)
        nxt = None
        if idx_in_all is not None and idx_in_all + 1 < len(all_toc_entries):
            nxt = all_toc_entries[idx_in_all + 1]

        # use page to extract
        body = extract_section_strict_by_printed_pages(pages_with_nums, t, nxt)

        sections.append({
            "section": t.get("section"),
            "heading": t.get("toc_title"),
            "text": body
        })

    return sections

### Read txt files

In [98]:
def candidate_filing_files(cik_dir: Path) -> List[Path]:
    exts = ('.txt', '.htm', '.html', '.sgm', '.sgml')
    return [f for f in cik_dir.rglob("*") if f.is_file() and f.suffix.lower() in exts]

### Process S-1

In [99]:
def drop_tables(s: str, placeholder: str = "") -> str:
    if not s:
        return s
    pat_closed = re.compile(r"(?is)<\s*table\b[^>]*>.*?</\s*table\s*>")  # match <table>...</table>
    out = pat_closed.sub(placeholder, s)
    return out

# replace <PAGE>\n with ""
def drop_page_markers(s: str) -> str:
    if not s:
        return s
    pat = re.compile(r"(?im)^[ \t]*<\s*PAGE\s*>[ \t]*(?:\r?\n|$)")
    return pat.sub("", s)


def process_one_file(chosen_file: Path) -> Dict:
    cik = chosen_file.stem
    result = {
        "cik": cik, 
        "status": "not_found", 
        "reason": "", 
        "s1_file": str(chosen_file), 
        "out_dir": None
    }

    raw = read_text_safely(chosen_file)
    text1 = re.sub(r"\r\n?", "\n", raw)
    text1 = drop_page_markers(text1)

    toc_block = extract_first_toc_block(text1)
    can_try_paged = toc_block is not None
    if not can_try_paged:
        result.update(status="garbled", reason="cannot find paged TOC")
        return result
    
    all_toc_entries = parse_toc_entries(toc_block)
    if not all_toc_entries:
        result.update(status="garbled", reason="paged TOC has no entries")
        return result
    
    # filter out target sections
    canon_targets_map = _build_canon_target_map(TARGET_SECTIONS, SYNONYMS)
    target_entries = []
    seen_sections = set() 

    for e in all_toc_entries:
        base, how = _pick_target_for_toc_title(e["toc_title"], canon_targets_map)
        if not base or base in seen_sections:      
            continue
        seen_sections.add(base)
        target_entries.append({
                "section": base,                 
                "toc_title": e["toc_title"],     
                "start_page": e["start_page"],
                "end_page": e.get("end_page"),
        })

    if not target_entries:
        result.update(status="garbled", reason="no target sections found in the paged TOC")
        return result
    
    try:
        sections = locate_sections(text1, all_toc_entries, target_entries)
    except RuntimeError: 
        sections = []

    if not sections or not any((s.get("text") or "").strip() for s in sections):
        result.update(status="garbled", reason="failed to locate sections by paged TOC")
        return result

    chapter_path = CHAPTER_DIR / f"{cik}.txt"
    with chapter_path.open("w", encoding="utf-8") as cf:
        for i, s in enumerate(sections, 1):
            s["text"] = drop_tables(s.get("text") or "", placeholder="")
            cf.write(f"{'#'*10} {i}. {s['section'].upper()} {'#'*10}\n")
            cf.write((s["text"] or "") + "\n\n")

    result.update(status="ok", chapter_file=str(chapter_path))
    return result

### Main

In [100]:
def main():
    #all_files = candidate_filing_files(S1_ROOT)
    #files = all_files[:10]
    files = candidate_filing_files(S1_ROOT)
    print(f"[INFO] read {len(files)} files under {S1_ROOT}")

    processed, garbled = [], []
    for f in files:
        print(f"\n[INFO] process FILE {f} ...")
        try:
            res = process_one_file(f)
        except Exception as e:
            print(f"[ERROR] {f} error: {e}")
            res = {"cik": f.stem, "status": "garbled", "reason": f"exception: {e}", "s1_file": str(f)}
        if res["status"] == "ok":
            processed.append(res)
            print(f"[OK] {f.name} CIK {res.get('cik')} -> chapter: {res.get('chapter_file')}")
        else:
            garbled.append(res)
            print(f"[WARN] {f.name} labeled as {res['status']}: {res.get('reason')}")
    if processed:
        pd.DataFrame(processed).to_excel(OUT_DIR / "section_extraction_processed_html_txt.xlsx", index=False)
        print(f"[OK] wrote processed list: {OUT_DIR/'section_extraction_processed_html_txt.xlsx'}")
    if garbled:
        pd.DataFrame(garbled).to_excel(OUT_DIR / "garbled_html_txt.xlsx", index=False)
        print(f"[OK] wrote garbled list: {OUT_DIR/'garbled_html_txt.xlsx'}")

if __name__ == "__main__":
    main()

[INFO] read 925 files under /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/htmlcode_txt/txt

[INFO] process FILE /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/htmlcode_txt/txt/0001361103.txt ...
[INFO] file /Users/panglinshao/Desktop/IPO/S-1/S-1 filings/htmlcode_txt/txt/0001361103.txt is successfully read using utf-8, length 592309
[TARGET] section=prospectus summary  heading='Prospectus Summary'  canon='prospectus summary'  start_page=1
[two-phase] phase-1 exact-line matched; skip grow. where=10163-10183 context='hip of us by, these other companies.\n\xa0\ni\nProspectus Summary\nThis summary highlights information cont'
[two-phase] phase-1 exact-line matched; skip grow. where=0-13 context='Risk Factors\nInvesting in our common stock involves a'
[TARGET] section=risk factors  heading='Risk Factors'  canon='risk factors'  start_page=12
[two-phase] phase-1 exact-line matched; skip grow. where=0-13 context='Risk Factors\nInvesting in our common stock involves a'
[two-phase] phase-1 exact